In [ ]:
# Standard libraries
import numpy as np
from pathlib import Path
from glob import glob
import plotly.express as px
import matplotlib.pyplot as plt
from tqdm import tqdm
from functools import partial
import pandas as pd
import json
from itertools import product

# Specific libraries
import nibabel as nib
from scipy.ndimage import distance_transform_edt
from scipy.stats import linregress
import seaborn as sns
import ants

# custom functions
import pp_utils as pp
import calib_utils as cal

In [ ]:
parent_dir = Path("./calibration")
deriv_dir = parent_dir / "derivatives"
testname = parent_dir.stem
plot_dir = parent_dir / "plots"
plot_dir.mkdir(exist_ok=True)

calib_dir = parent_dir / "calib"
calib_dir.mkdir(exist_ok=True)

param_fname = deriv_dir / (testname + "_" + "params.h5")
scan_fname = deriv_dir / (testname + "_" + "RAW_MPPCA_REG.nii.gz")
mask_fname = deriv_dir / (testname + "_" + "MASK.nii.gz")

# Load in sequence parameters
df = pd.read_hdf(param_fname, key="df", mode="r")

# Load in scan data and add into data frame
scan_nii = nib.load(scan_fname)
scan_data = scan_nii.get_fdata()
scan_aff = scan_nii.affine
scan_spacing = pp.spacing_from_affine(scan_aff)

scan_data_reshuffled = np.transpose(scan_data,[3,0,1,2])
df["image"] = list(scan_data_reshuffled)
df["adc"] = df["temperature"].apply(partial(pp.water_diffusivity, polynomial=True))

# Load in mask
mask_nii = nib.load(mask_fname)
mask = mask_nii.get_fdata().astype(int)

# Create a mask slightly inset from threshold based mask to avoid boundary artefacts.
shrink_dist = 6 # mm
core_mask = cal.shrink_mask_mm(mask, voxel_spacing=scan_spacing, margin_mm=shrink_dist)

In [ ]:
df.sort_values("scan_time", inplace=True, ignore_index=True)

# Put the b=0 image on b=0 rows, NaN elsewhere
b0_threshold = 5
df["s0_image"] = df["image"].where(df["b_value"] < b0_threshold)

# Forward-fill so each DWI scan gets the most recent preceding b=0 image
df["s0_image"] = df["s0_image"].ffill()

# Keep only diffusion-weighted scans
dwi_df = df[df["b_value"] > b0_threshold].copy()

# Optional safety check: drop rows that do not have a preceding b=0 scan
dwi_df = dwi_df[dwi_df["s0_image"].notna()].copy()

dwi_df["bmap"] = dwi_df.apply(lambda row: cal.bmap_3d(s0=row["s0_image"], 
                                                      s=row["image"], 
                                                      adc=row["adc"], 
                                                      mask=core_mask), 
                                                      axis=1)
dwi_df["cmap"] = dwi_df["bmap"]/dwi_df["b_value"]
# dwi_df["cmap_bspline"] = dwi_df.apply(lambda row: cal.bspline_estimation(data=row["cmap"], spacing=row["spacing"], mask=core_mask), axis=1)

# Estimate gradient maps from b-value maps assuming Stejskal-Tanner Equation applies.
dwi_df["gmap2"] = dwi_df.apply(lambda row: cal.b2grad2(b=row["bmap"],
                                              delta_small=row["dwi_grad_duration"],
                                              delta_big=(row["TE"] / 2),
                                              mask=core_mask),
                                              axis=1)

# Estimate a smoothed version of each gradient map for fitting
dwi_df["gmap2_bspline"] = dwi_df.apply(lambda row: cal.bspline_estimation(data=row["gmap2"], spacing=row["spacing"], mask=core_mask), axis=1)

In [ ]:
def bmap_estimation_error_loo(df, use_avg=False, verbose=False):

    loo_df = df.copy()
    loo_df["repeat_idx"] = loo_df.groupby("seq_name").cumcount()

    # Number of repeats available per sequence
    repeat_counts = loo_df.groupby("seq_name").size()
    n_repeats = repeat_counts.min()

    if repeat_counts.nunique() != 1:
        print("Warning: not all seq_name groups have the same number of repeats.")
        print("Using n_repeats =", n_repeats, "(minimum count across seq_name groups)")

    # Restrict to complete repeats only if some sequences have fewer repeats
    loo_df = loo_df[loo_df["repeat_idx"] < n_repeats].copy()
    seq_order = sorted(loo_df["seq_name"].unique())

    # Storage across folds
    error_map_store = []

    # -----------------------------------------------------------------------------
    # Leave-one-out across repeat index
    # -----------------------------------------------------------------------------
    for loo_idx in range(n_repeats):
        
        if verbose:
            print(f"LOO fold {loo_idx+1}/{n_repeats}")

        test_df = loo_df[loo_df["repeat_idx"] == loo_idx].copy()
        train_df = loo_df[loo_df["repeat_idx"] != loo_idx].copy()

        # Ensure consistent ordering
        test_df["seq_name"] = pd.Categorical(test_df["seq_name"], categories=seq_order, ordered=True)
        train_df["seq_name"] = pd.Categorical(train_df["seq_name"], categories=seq_order, ordered=True)

        test_df = test_df.sort_values("seq_name").reset_index(drop=True)
        train_df = train_df.sort_values("seq_name").reset_index(drop=True)
        eval_bmaps = np.stack(test_df["bmap"].values, axis=-1).astype(np.float32)

        if use_avg:

            pred_bmaps = []
            for seq in seq_order:
                sub_df = train_df[train_df["seq_name"] == seq].copy()
                seq_bmaps = np.stack(sub_df["bmap"].values, axis=-1)
                seq_bmap_avg = np.mean(seq_bmaps, axis=-1)
                pred_bmaps.append(seq_bmap_avg)
            pred_bmaps = np.stack(pred_bmaps, axis=-1)
            
        else:

            # Train calibration model
            train_gvecs = np.stack(train_df["grad_vec"].values, axis=0).astype(np.float32)
            train_gmaps = np.stack(train_df["gmap2_bspline"].values, axis=-1).astype(np.float32)
            A, sigma = cal.calc_calib_maps(train_gvecs, train_gmaps, core_mask, as_image=False)

            # Test on held-out repeat
            test_gvecs = np.stack(test_df["grad_vec"].values, axis=0).astype(np.float32)
            delta_small = test_df["dwi_grad_duration"].to_numpy(dtype=np.float32).reshape(1, 1, 1, -1)
            delta_big = ((test_df["TE"] / 2)).to_numpy(dtype=np.float32).reshape(1, 1, 1, -1)

            test_X = np.apply_along_axis(cal.x_from_gvec, arr=test_gvecs, axis=1)
            test_Z = np.apply_along_axis(partial(cal.z_from_x, sigma=sigma), arr=test_X, axis=1)

            pred_g2maps = np.zeros_like(eval_bmaps, dtype=np.float32)
            pred_g2maps[core_mask == 1] = (test_Z @ A).T
            pred_bmaps = cal.grad22b(pred_g2maps, delta_small=delta_small, delta_big=delta_big, mask=core_mask)

        error_matrix = eval_bmaps[core_mask==1] / pred_bmaps[core_mask==1]
        error_map_store.append(error_matrix)

    error_map_store_np = np.stack(error_map_store, axis=-1)
    error_vec = error_map_store_np.ravel()

    return error_vec

In [ ]:
# Estimate the Error for the traditional case
trad_df = dwi_df.copy()
trad_df["e_map"] = trad_df.apply(lambda row: np.nan_to_num(row["bmap"] / row["b_value"], nan=0, posinf=0, neginf=0), axis=1)
trad_error_matrix = np.stack(trad_df["e_map"].values, axis=-1)
trad_error_vec = trad_error_matrix[core_mask==1].ravel()

In [ ]:
# Error from full scanner bmap estimation
print("Full scan bmap estimation")
scanner_specific_error_vec = bmap_estimation_error_loo(dwi_df)

# Error from TE specific bmap estimation
dur_specific_error_store = []
for dur in [0.03, 0.035, 0.04]:

    print(rf"TE-specific bmap estimation: dur = {dur * 1e3:.0f} ms")
    dur_df = dwi_df[dwi_df["dwi_grad_duration"]==dur].copy()

    dur_iter_error_vec = bmap_estimation_error_loo(dur_df)
    dur_specific_error_store.append(dur_iter_error_vec)

dur_specific_error_matrix = np.stack(dur_specific_error_store, axis=-1)
dur_specific_error_vec = dur_specific_error_matrix.ravel()

# Error from sequence specific bmap estimation
seq_specific_error_store = []
for seq in tqdm(dwi_df["seq_name"].unique(), desc="Sequence specific bmap estimation"):

    seq_df = dwi_df[dwi_df["seq_name"]==seq].copy()

    seq_iter_error_vec = bmap_estimation_error_loo(seq_df, use_avg=True, verbose=False)
    seq_specific_error_store.append(seq_iter_error_vec)

seq_specific_error_matrix = np.stack(seq_specific_error_store, axis=-1)
seq_specific_error_vec = seq_specific_error_matrix.ravel()

In [ ]:
# Long-form dataframe
n_test = 4
df = pd.DataFrame({
    "Error": np.concatenate([
        trad_error_vec,
        scanner_specific_error_vec,
        dur_specific_error_vec,
        seq_specific_error_vec
    ]),
    "Group": (
        ["No correction"] * len(trad_error_vec) + 
        ["Scanner specific"] * len(scanner_specific_error_vec) +
        ["TE specific"] * len(dur_specific_error_vec) +
        ["Sequence specific"] * len(seq_specific_error_vec)
    )
})

order = [
    "No correction", 
    "Scanner specific", 
    "TE specific", 
    "Sequence specific"]
wrapped_order = [
    "No\ncorrection",
    "Scanner\nlevel\ncorrection",
    "Duration\nlevel\ncorrection",
    "Sequence\nlevel\ncorrection"]

rocket = sns.color_palette("rocket", n_test+1)
palette = [rocket[i+1] for i in range(n_test)]

fig, ax = plt.subplots(figsize=(5.5, 4.5))
fs_labels = 10
plt.grid(visible=True, axis="y", c="grey", linewidth=1, alpha=0.25)

ax.axhline(0, color="0.35", linestyle="--", c="grey", linewidth=1)
sns.boxplot(
    data=df,
    x="Group",
    y="Error",
    hue="Group",
    order=order,
    width=0.5,
    showfliers=False,      # important for very large datasets
    whis=(5, 95),          # often more stable/readable than default 1.5*IQR here
    palette=palette,
    linewidth=1.2,
    boxprops=dict(alpha=1),
    medianprops=dict(color="black", linewidth=1.2),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2),
    ax=ax
)

ax.set_xlabel("")
ax.set_ylim(0.6, 1.3)
ax.set_xticklabels(wrapped_order, fontsize=fs_labels, style='italic')
ax.set_ylabel(r"c(r)", fontsize=fs_labels)

plt.tight_layout()

fmt = "png"
plot_fname = plot_dir / (testname + "_" + f"calibration_framework_comparison_boxplot_correction_factor.{fmt}")
fig.savefig(fname=plot_fname, dpi=600, format=fmt, bbox_inches="tight", pad_inches=0.01)

In [ ]:
group_metrics = df.groupby("Group")["Error"].agg(mean="mean", 
                                                  std="std", 
                                                  median="median", 
                                                  q1=lambda x: x.quantile(0.25), 
                                                  q2=lambda x: x.quantile(0.75),
                                                  iqr=lambda x: x.quantile(0.75) - x.quantile(0.25),
                                                  iwr=lambda x: x.quantile(0.95) - x.quantile(0.05))
metrics_fname = plot_dir / (testname + "_" + "calibration_framework_compariscon_metrics.csv")
group_metrics.to_csv(metrics_fname, sep=",", header=True, index=True)